In [8]:
from src.spark import spark_builder
from src.paths import BRONZE_DIR, s

spark = spark_builder("bronze_inspect")

Bronze_df_path = BRONZE_DIR / "onlineRetailBronze"

In [9]:
df = spark.read.parquet(s(Bronze_df_path))
df.show(5, truncate=False)

+---------+---------+-----------------------------------+--------+-------------+---------+----------+--------------+------------------------------------------------------------------------------------+--------------------------+
|InvoiceNo|StockCode|Description                        |Quantity|InvoiceDate  |UnitPrice|CustomerID|Country       |_source_file                                                                        |_ingest_date              |
+---------+---------+-----------------------------------+--------+-------------+---------+----------+--------------+------------------------------------------------------------------------------------+--------------------------+
|545082   |21689    |SILVER VANILLA  FLOWER CANDLE POT  |4       |2/28/11 10:37|3.75     |16992     |United Kingdom|file:///home/vscarter/Desktop/Repos/online-retail-pipline/data/raw/Online_Retail.csv|2026-02-05 13:53:11.508483|
|545082   |21688    |SILVER PLATE CANDLE BOWL SMALL     |6       |2/28/11 10:37|2.95

In [10]:
df.printSchema()

root
 |-- InvoiceNo: string (nullable = true)
 |-- StockCode: string (nullable = true)
 |-- Description: string (nullable = true)
 |-- Quantity: string (nullable = true)
 |-- InvoiceDate: string (nullable = true)
 |-- UnitPrice: string (nullable = true)
 |-- CustomerID: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- _source_file: string (nullable = true)
 |-- _ingest_date: timestamp (nullable = true)



In [11]:
df.count()

541909

In [12]:
len(df.columns)

10

In [13]:
from pyspark.sql import functions as F

unique_count = df.select([
    F.countDistinct(F.col(c)).alias(c)
    for c in df.columns
])

unique_count.show(truncate=False)

+---------+---------+-----------+--------+-----------+---------+----------+-------+------------+------------+
|InvoiceNo|StockCode|Description|Quantity|InvoiceDate|UnitPrice|CustomerID|Country|_source_file|_ingest_date|
+---------+---------+-----------+--------+-----------+---------+----------+-------+------------+------------+
|25900    |4070     |4223       |722     |23260      |1630     |4372      |38     |1           |1           |
+---------+---------+-----------+--------+-----------+---------+----------+-------+------------+------------+



In [27]:
inv_country = (
    df.groupBy("Country", "CustomerID")
    .agg(
        F.count("*").alias("Purchases"),
        F.format_number(F.sum("UnitPrice"),2).alias("Total_revenue"),
        F.format_number(F.avg("Quantity"),2).alias("average_amount_bought")
    )
    .orderBy("Country", ascending=True)
)

inv_country.count()
inv_country.show()

+---------+----------+---------+-------------+---------------------+
|  Country|CustomerID|Purchases|Total_revenue|average_amount_bought|
+---------+----------+---------+-------------+---------------------+
|Australia|     12434|       54|       198.13|                 6.91|
|Australia|     12415|      778|     2,499.82|                99.28|
|Australia|     12388|      100|       277.77|                14.62|
|Australia|     12393|       64|       145.90|                12.75|
|Australia|     12431|      186|       718.08|                12.87|
|Australia|     12424|       30|        83.42|                24.67|
|Australia|     12422|       21|        51.12|                 9.29|
|Australia|     16321|       16|        56.60|                 4.88|
|Australia|     12386|       10|        23.91|                35.40|
|  Austria|     12429|       21|       114.33|                 9.43|
|  Austria|     12414|       18|       154.26|                16.67|
|  Austria|     12360|      129|  

In [29]:
null_count = df.select([
    F.sum(F.col(c).isNull().cast("int")).alias(c)
    for c in df.columns
])

null_count.show()

+---------+---------+-----------+--------+-----------+---------+----------+-------+------------+------------+
|InvoiceNo|StockCode|Description|Quantity|InvoiceDate|UnitPrice|CustomerID|Country|_source_file|_ingest_date|
+---------+---------+-----------+--------+-----------+---------+----------+-------+------------+------------+
|        0|        0|       1454|       0|          0|        0|    135080|      0|           0|           0|
+---------+---------+-----------+--------+-----------+---------+----------+-------+------------+------------+



In [30]:
len(df.columns)

10